In [20]:
import os
import pandas as pd
import numpy as np

N_FOLDS    = 3
TEST_NAMES = ['lfw', 'cfp_fp', 'agedb_30', 'sllfw', 'talfw']  # ← adjust if needed

from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    f1_score, accuracy_score, roc_curve
)

def compute_from_csv(csv_path):
    df        = pd.read_csv(csv_path)
    probs     = df['prob'].values.flatten() if 'prob' in df.columns else df['prob_same'].values.flatten()
    labels    = df['label'].values.flatten()

    # use optimal threshold from val calibration if available, else 0.5
    threshold = df['threshold'].iloc[0] if 'threshold' in df.columns else 0.5
    preds     = (probs >= threshold).astype(int)

    return {
        'acc' : accuracy_score(labels, preds),
        'f1'  : f1_score(labels, preds, average='macro'),
        'auc' : roc_auc_score(labels, probs),
        'ap'  : average_precision_score(labels, probs),
    }

def compute_metrics(log_name, calibrated=False):
    BASE_DIR   = f'tb_logs/{log_name}_face'
    
    rows = []
    for fold in range(N_FOLDS):
        fold_dir = os.path.join(BASE_DIR, f'fold_{fold}')
        for name in TEST_NAMES:
            csv_path = os.path.join(fold_dir, f'{name}_{"calibrated_" if calibrated else ""}predictions.csv')
            if not os.path.exists(csv_path):
                print(f'[WARN] missing: {csv_path}')
                continue
            metrics = compute_from_csv(csv_path)
            rows.append({'fold': fold, 'benchmark': name, **metrics})

    df = pd.DataFrame(rows)

    # ── aggregate: mean ± std per benchmark ──────────────────────────────────────
    agg = df.groupby('benchmark')[['acc', 'f1', 'auc', 'ap']].agg(['mean', 'std'])

    # flatten column names: (acc, mean) → acc_mean
    agg.columns = ['_'.join(c) for c in agg.columns]
    agg = agg.reindex(TEST_NAMES)   # preserve benchmark order

    print(f"\n── {log_name} — cross-fold summary (n={N_FOLDS} folds) ──────────────────")
    print(f"\n{'benchmark':<12} {'ACC':>20} {'F1':>17} {'AUC':>17} {'AP':>17}")
    print("─" * 92)
    for bm, row in agg.iterrows():
        print(
            f"{bm:<12}      "
            f"{row['acc_mean']:.4f} ± {row['acc_std']:.4f}   "
            f"{row['f1_mean']:.4f} ± {row['f1_std']:.4f}   "
            f"{row['auc_mean']:.4f} ± {row['auc_std']:.4f}   "
            f"{row['ap_mean']:.4f} ± {row['ap_std']:.4f}"
        )
        
    out_path = os.path.join(BASE_DIR, f'{"calibrated_" if calibrated else ""}cross_fold_summary.csv')
    agg.to_csv(out_path)
    print(f"\nSaved to: {out_path}")

In [21]:
compute_metrics('simple_cnn_contrastive')


── simple_cnn_contrastive — cross-fold summary (n=3 folds) ──────────────────

benchmark                     ACC                F1               AUC                AP
────────────────────────────────────────────────────────────────────────────────────────────
lfw               0.5921 ± 0.0080   0.5106 ± 0.0135   0.9051 ± 0.0033   0.8885 ± 0.0024
cfp_fp            0.5626 ± 0.0125   0.4601 ± 0.0234   0.7769 ± 0.0225   0.7369 ± 0.0254
agedb_30          0.5897 ± 0.0223   0.5078 ± 0.0375   0.7040 ± 0.0192   0.6490 ± 0.0121
sllfw             0.5001 ± 0.0001   0.3336 ± 0.0002   0.6274 ± 0.0075   0.6174 ± 0.0091
talfw             0.5793 ± 0.0060   0.4888 ± 0.0104   0.8668 ± 0.0016   0.8429 ± 0.0054

Saved to: tb_logs/simple_cnn_contrastive_face\cross_fold_summary.csv


In [22]:
compute_metrics('simple_cnn_arcface')


── simple_cnn_arcface — cross-fold summary (n=3 folds) ──────────────────

benchmark                     ACC                F1               AUC                AP
────────────────────────────────────────────────────────────────────────────────────────────
lfw               0.9015 ± 0.0016   0.9011 ± 0.0017   0.9733 ± 0.0014   0.9749 ± 0.0015
cfp_fp            0.6718 ± 0.0053   0.6388 ± 0.0075   0.8670 ± 0.0047   0.8690 ± 0.0058
agedb_30          0.6213 ± 0.0062   0.5811 ± 0.0079   0.7788 ± 0.0059   0.7611 ± 0.0094
sllfw             0.6654 ± 0.0004   0.6543 ± 0.0008   0.7881 ± 0.0022   0.8132 ± 0.0024
talfw             0.8114 ± 0.0024   0.8092 ± 0.0024   0.9121 ± 0.0019   0.9149 ± 0.0019

Saved to: tb_logs/simple_cnn_arcface_face\cross_fold_summary.csv


In [23]:
compute_metrics('simple_cnn_arcface', calibrated=True)


── simple_cnn_arcface — cross-fold summary (n=3 folds) ──────────────────

benchmark                     ACC                F1               AUC                AP
────────────────────────────────────────────────────────────────────────────────────────────
lfw               0.8687 ± 0.0023   0.8672 ± 0.0024   0.9733 ± 0.0014   0.9749 ± 0.0015
cfp_fp            0.7848 ± 0.0065   0.7847 ± 0.0065   0.8671 ± 0.0046   0.8691 ± 0.0058
agedb_30          0.7015 ± 0.0062   0.7009 ± 0.0063   0.7784 ± 0.0061   0.7608 ± 0.0096
sllfw             0.5338 ± 0.0004   0.4217 ± 0.0024   0.7882 ± 0.0022   0.8134 ± 0.0024
talfw             0.7888 ± 0.0011   0.7841 ± 0.0010   0.9122 ± 0.0019   0.9150 ± 0.0020

Saved to: tb_logs/simple_cnn_arcface_face\calibrated_cross_fold_summary.csv


In [24]:
compute_metrics('effnet_contrastive')


── effnet_contrastive — cross-fold summary (n=3 folds) ──────────────────

benchmark                     ACC                F1               AUC                AP
────────────────────────────────────────────────────────────────────────────────────────────
lfw               0.6475 ± 0.0121   0.5974 ± 0.0177   0.9305 ± 0.0025   0.9107 ± 0.0038
cfp_fp            0.6832 ± 0.0120   0.6489 ± 0.0167   0.8787 ± 0.0015   0.8457 ± 0.0015
agedb_30          0.6354 ± 0.0140   0.5818 ± 0.0211   0.7234 ± 0.0072   0.6577 ± 0.0093
sllfw             0.5006 ± 0.0004   0.3350 ± 0.0006   0.6447 ± 0.0035   0.6173 ± 0.0031
talfw             0.6083 ± 0.0071   0.5397 ± 0.0127   0.7856 ± 0.0084   0.7372 ± 0.0108

Saved to: tb_logs/effnet_contrastive_face\cross_fold_summary.csv


In [25]:
compute_metrics('effnet_arcface')


── effnet_arcface — cross-fold summary (n=3 folds) ──────────────────

benchmark                     ACC                F1               AUC                AP
────────────────────────────────────────────────────────────────────────────────────────────
lfw               0.7391 ± 0.0060   0.7201 ± 0.0074   0.9885 ± 0.0003   0.9899 ± 0.0005
cfp_fp            0.5128 ± 0.0010   0.3611 ± 0.0021   0.9035 ± 0.0010   0.9130 ± 0.0020
agedb_30          0.5116 ± 0.0009   0.3587 ± 0.0019   0.8269 ± 0.0059   0.8243 ± 0.0048
sllfw             0.7331 ± 0.0068   0.7146 ± 0.0087   0.9021 ± 0.0028   0.9161 ± 0.0023
talfw             0.5428 ± 0.0023   0.4242 ± 0.0047   0.7672 ± 0.0062   0.7770 ± 0.0051

Saved to: tb_logs/effnet_arcface_face\cross_fold_summary.csv


In [26]:
compute_metrics('effnet_arcface', calibrated=True)


── effnet_arcface — cross-fold summary (n=3 folds) ──────────────────

benchmark                     ACC                F1               AUC                AP
────────────────────────────────────────────────────────────────────────────────────────────
lfw               0.9061 ± 0.0023   0.9055 ± 0.0023   0.9885 ± 0.0003   0.9899 ± 0.0005
cfp_fp            0.8289 ± 0.0018   0.8287 ± 0.0018   0.9034 ± 0.0009   0.9130 ± 0.0018
agedb_30          0.7520 ± 0.0058   0.7516 ± 0.0056   0.8267 ± 0.0061   0.8242 ± 0.0050
sllfw             0.6126 ± 0.0049   0.5516 ± 0.0076   0.9023 ± 0.0029   0.9163 ± 0.0024
talfw             0.6858 ± 0.0050   0.6815 ± 0.0045   0.7669 ± 0.0063   0.7767 ± 0.0050

Saved to: tb_logs/effnet_arcface_face\calibrated_cross_fold_summary.csv


# Buffalo from Insight Face

```python

── Buffalo S ──────────────────

benchmark                     ACC                F1               AUC                AP
────────────────────────────────────────────────────────────────────────────────────────────
lfw                                                            0.9993            0.9995
cfp_fp                                                         0.9738            0.9831
agedb_30                                                       0.9892            0.9921
sllfw                                                          0.9976            0.9985
talfw                                                          0.5863            0.5856



── Buffalo L ──────────────────

benchmark                     ACC                F1               AUC                AP
────────────────────────────────────────────────────────────────────────────────────────────
lfw                                                            0.9994            0.9996
cfp_fp                                                         0.9800            0.9871
agedb_30                                                       0.9912            0.9943
sllfw                                                          0.9985            0.9991
talfw                                                          0.7486            0.7588
```